In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Raw CSV**

In [ ]:
import os
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/bigdata-gbl-quiz"
RAW_FILE = os.path.join(PROJECT_ROOT, "data", "raw", "quiz_interactions.csv")
PROCESSED_FILE = os.path.join(PROJECT_ROOT, "data", "processed", "quiz_features.csv")

df = pd.read_csv(RAW_FILE)
df.head(), df.shape

(  learner_id question_id      topic  difficulty  time_spent_seconds  \
 0      L0202       Q0533     Sports           1                  47   
 1      L0227       Q0249    History           1                  37   
 2      L0115       Q0472     Sports           3                  91   
 3      L0300       Q0506  Geography           2                  67   
 4      L0150       Q0195     Sports           1                  43   
 
    attempts_count  hints_used  correct  
 0               2           0        0  
 1               2           0        1  
 2               3           2        0  
 3               2           1        0  
 4               2           1        1  ,
 (20000, 8))

In [ ]:
print("Data types:\n", df.dtypes)
print("\nMissing values:\n", df.isna().sum())

Data types:
 learner_id            object
question_id           object
topic                 object
difficulty             int64
time_spent_seconds     int64
attempts_count         int64
hints_used             int64
correct                int64
dtype: object

Missing values:
 learner_id            0
question_id           0
topic                 0
difficulty            0
time_spent_seconds    0
attempts_count        0
hints_used            0
correct               0
dtype: int64


In [ ]:
df_clean = df.copy()

df_clean["difficulty"] = df_clean["difficulty"].astype(int)
df_clean["time_spent_seconds"] = df_clean["time_spent_seconds"].astype(int)
df_clean["attempts_count"] = df_clean["attempts_count"].astype(int)
df_clean["hints_used"] = df_clean["hints_used"].astype(int)
df_clean["correct"] = df_clean["correct"].astype(int)

# Keep everything in sensible ranges (safety)
df_clean["time_spent_seconds"] = df_clean["time_spent_seconds"].clip(5, 240)
df_clean["attempts_count"] = df_clean["attempts_count"].clip(1, 10)
df_clean["hints_used"] = df_clean["hints_used"].clip(0, 5)

# Remove duplicate rows if any
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)

print("Rows before:", before)
print("Rows after removing duplicates:", after)

Rows before: 20000
Rows after removing duplicates: 19991


In [ ]:
df_clean["time_per_attempt"] = df_clean["time_spent_seconds"] / df_clean["attempts_count"]
df_clean[["time_spent_seconds", "attempts_count", "time_per_attempt"]].head()

,time_spent_seconds,attempts_count,time_per_attempt
0,47,2,23.500000
1,37,2,18.500000
2,91,3,30.333333
3,67,2,33.500000
4,43,2,21.500000


In [ ]:
df_clean["attempt_index"] = df_clean.groupby("learner_id").cumcount() + 1
df_clean[["learner_id", "attempt_index"]].head(10)

,learner_id,attempt_index
0,L0202,1
1,L0227,1
2,L0115,1
3,L0300,1
4,L0150,1
5,L0049,1
6,L0002,1
7,L0061,1
8,L0208,1
9,L0162,1


In [ ]:
df_clean = df_clean.sort_values(["learner_id", "attempt_index"]).reset_index(drop=True)
df_clean.head()

,learner_id,question_id,topic,difficulty,time_spent_seconds,attempts_count,hints_used,correct,time_per_attempt,attempt_index
0,L0001,Q0103,Geography,2,73,3,2,0,24.333333,1
1,L0001,Q0561,History,2,74,3,1,0,24.666667,2
2,L0001,Q0189,ArtsCulture,1,37,2,0,1,18.500000,3
3,L0001,Q0443,ArtsCulture,1,60,2,1,0,30.000000,4
4,L0001,Q0392,Technology,1,33,2,0,1,16.500000,5


In [ ]:
df_clean["recent_success_5"] = (
    df_clean.groupby("learner_id")["correct"]
    .apply(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

df_clean[["learner_id", "attempt_index", "correct", "recent_success_5"]].head(12)

,learner_id,attempt_index,correct,recent_success_5
0,L0001,1,0,NaN
1,L0001,2,0,0.000000
2,L0001,3,1,0.000000
3,L0001,4,0,0.333333
4,L0001,5,1,0.250000
5,L0001,6,1,0.400000
6,L0001,7,0,0.600000
7,L0001,8,0,0.600000
8,L0001,9,0,0.400000
9,L0001,10,1,0.400000


In [ ]:
df_features = df_clean.copy()

df_features["recent_success_5"] = df_features["recent_success_5"].fillna(0.5)

print("Any NaNs left?", df_features["recent_success_5"].isna().sum())

Any NaNs left? 0


 Save the processed dataset

In [ ]:
df_features.to_csv(PROCESSED_FILE, index=False)
print("✅ Saved processed file to:", PROCESSED_FILE)

# Reload to confirm it saved properly
df_check = pd.read_csv(PROCESSED_FILE)
df_check.shape, df_check.head()

✅ Saved processed file to: /content/drive/MyDrive/bigdata-gbl-quiz/data/processed/quiz_features.csv


((19991, 11),
   learner_id question_id        topic  difficulty  time_spent_seconds  \
 0      L0001       Q0103    Geography           2                  73   
 1      L0001       Q0561      History           2                  74   
 2      L0001       Q0189  ArtsCulture           1                  37   
 3      L0001       Q0443  ArtsCulture           1                  60   
 4      L0001       Q0392   Technology           1                  33   
 
    attempts_count  hints_used  correct  time_per_attempt  attempt_index  \
 0               3           2        0         24.333333              1   
 1               3           1        0         24.666667              2   
 2               2           0        1         18.500000              3   
 3               2           1        0         30.000000              4   
 4               2           0        1         16.500000              5   
 
    recent_success_5  
 0          0.500000  
 1          0.000000  
 2          0

In [ ]:
df_features.loc[df_features["learner_id"]=="L0001",
                ["attempt_index","correct","recent_success_5"]].head(8)

,attempt_index,correct,recent_success_5
0,1,0,0.500000
1,2,0,0.000000
2,3,1,0.000000
3,4,0,0.333333
4,5,1,0.250000
5,6,1,0.400000
6,7,0,0.600000
7,8,0,0.600000


In [ ]:
df_clean["recent_hints_5"] = (
    df_clean.groupby("learner_id")["hints_used"]
    .apply(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

df_clean[["learner_id", "attempt_index", "hints_used", "recent_hints_5"]].head(12)

,learner_id,attempt_index,hints_used,recent_hints_5
0,L0001,1,2,NaN
1,L0001,2,1,2.0
2,L0001,3,0,1.5
3,L0001,4,1,1.0
4,L0001,5,0,1.0
5,L0001,6,0,0.8
6,L0001,7,0,0.4
7,L0001,8,1,0.2
8,L0001,9,0,0.4
9,L0001,10,1,0.2


In [ ]:
df_clean["recent_time_5"] = (
    df_clean.groupby("learner_id")["time_spent_seconds"]
    .apply(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

df_clean[["learner_id", "attempt_index", "time_spent_seconds", "recent_time_5"]].head(12)

,learner_id,attempt_index,time_spent_seconds,recent_time_5
0,L0001,1,73,NaN
1,L0001,2,74,73.000000
2,L0001,3,37,73.500000
3,L0001,4,60,61.333333
4,L0001,5,33,61.000000
5,L0001,6,20,55.400000
6,L0001,7,17,44.800000
7,L0001,8,26,33.400000
8,L0001,9,41,31.200000
9,L0001,10,44,27.400000


**Questions Completed**

In [ ]:
df_clean["questions_completed"] = df_clean["attempt_index"]

**Average difficulty so far**

In [ ]:
df_clean["avg_difficulty_so_far"] = (
    df_clean.groupby("learner_id")["difficulty"]
    .apply(lambda s: s.shift(1).expanding(min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

In [ ]:
df_clean[["learner_id","attempt_index","difficulty","avg_difficulty_so_far"]].head(12)

,learner_id,attempt_index,difficulty,avg_difficulty_so_far
0,L0001,1,2,NaN
1,L0001,2,2,2.000000
2,L0001,3,1,2.000000
3,L0001,4,1,1.666667
4,L0001,5,1,1.500000
5,L0001,6,1,1.400000
6,L0001,7,1,1.333333
7,L0001,8,1,1.285714
8,L0001,9,2,1.250000
9,L0001,10,1,1.333333


In [ ]:
df_features = df_clean.copy()

df_features["recent_success_5"] = df_features["recent_success_5"].fillna(0.5)
df_features["recent_hints_5"] = df_features["recent_hints_5"].fillna(0.0)
df_features["recent_time_5"] = df_features["recent_time_5"].fillna(df_features["time_spent_seconds"].median())
df_features["avg_difficulty_so_far"] = df_features["avg_difficulty_so_far"].fillna(df_features["difficulty"].median())

print("NaNs left?\n", df_features[["recent_success_5","recent_hints_5","recent_time_5","avg_difficulty_so_far"]].isna().sum())

NaNs left?
 recent_success_5         0
recent_hints_5           0
recent_time_5            0
avg_difficulty_so_far    0
dtype: int64


In [ ]:
df_features.to_csv(PROCESSED_FILE, index=False)
print("✅ Saved updated processed dataset to:", PROCESSED_FILE)

df_check = pd.read_csv(PROCESSED_FILE)
print("Shape:", df_check.shape)
df_check.head()

✅ Saved updated processed dataset to: /content/drive/MyDrive/bigdata-gbl-quiz/data/processed/quiz_features.csv
Shape: (19991, 15)


,learner_id,question_id,topic,difficulty,time_spent_seconds,attempts_count,hints_used,correct,time_per_attempt,attempt_index,recent_success_5,recent_hints_5,recent_time_5,questions_completed,avg_difficulty_so_far
0,L0001,Q0103,Geography,2,73,3,2,0,24.333333,1,0.500000,0.0,56.000000,1,2.000000
1,L0001,Q0561,History,2,74,3,1,0,24.666667,2,0.000000,2.0,73.000000,2,2.000000
2,L0001,Q0189,ArtsCulture,1,37,2,0,1,18.500000,3,0.000000,1.5,73.500000,3,2.000000
3,L0001,Q0443,ArtsCulture,1,60,2,1,0,30.000000,4,0.333333,1.0,61.333333,4,1.666667
4,L0001,Q0392,Technology,1,33,2,0,1,16.500000,5,0.250000,1.0,61.000000,5,1.500000


In [ ]:
df_features.loc[df_features["learner_id"]=="L0001",
                ["attempt_index","correct","recent_success_5","recent_hints_5","recent_time_5","avg_difficulty_so_far"]].head(8)

,attempt_index,correct,recent_success_5,recent_hints_5,recent_time_5,avg_difficulty_so_far
0,1,0,0.500000,0.0,56.000000,2.000000
1,2,0,0.000000,2.0,73.000000,2.000000
2,3,1,0.000000,1.5,73.500000,2.000000
3,4,0,0.333333,1.0,61.333333,1.666667
4,5,1,0.250000,1.0,61.000000,1.500000
5,6,1,0.400000,0.8,55.400000,1.400000
6,7,0,0.600000,0.4,44.800000,1.333333
7,8,0,0.600000,0.2,33.400000,1.285714
